In [25]:
import numpy as np
import pandas as pd


In [26]:
import joblib
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error


In [27]:
data = pd.read_csv("car_ads_details_kaggle.csv")


In [28]:
print(data.info())
print(data.isnull().sum())
print(data.duplicated().sum())


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8374 entries, 0 to 8373
Data columns (total 9 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   Brand                 8372 non-null   object 
 1   Model                 8316 non-null   object 
 2   Kilometers            8373 non-null   object 
 3   Year                  8374 non-null   int64  
 4   Fuel Type             8374 non-null   object 
 5   Transmission Type     8374 non-null   object 
 6   Engine Capacity (CC)  6399 non-null   float64
 7   Body Type             7819 non-null   object 
 8   Price_EGP             8374 non-null   object 
dtypes: float64(1), int64(1), object(7)
memory usage: 588.9+ KB
None
Brand                      2
Model                     58
Kilometers                 1
Year                       0
Fuel Type                  0
Transmission Type          0
Engine Capacity (CC)    1975
Body Type                555
Price_EGP                  0
dty

In [29]:
data = data.drop_duplicates()


In [30]:
data["Price_EGP"] = (
    data["Price_EGP"]
    .astype(str)
    .str.replace("EGP", "")
    .str.replace(",", "")
    .str.strip()
    .astype(int)
)


In [31]:
data["Kilometers"] = pd.to_numeric(data["Kilometers"], errors="coerce")


In [32]:
text_columns = ["Brand", "Model", "Fuel Type", "Transmission Type", "Body Type"]
for col in text_columns:
    data[col] = data[col].str.strip().str.title()


In [33]:
data.loc[(data["Year"] < 1970) | (data["Year"] > 2026), "Year"] = np.nan


##(Feature Engineering)

In [34]:
data["Car_Age"] = 2026 - data["Year"]
data = data.drop("Year", axis=1)


In [35]:
X = data.drop("Price_EGP", axis=1)
y = data["Price_EGP"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)


In [36]:
median_km = X_train["Kilometers"].median()
median_engine = X_train["Engine Capacity (CC)"].median()
most_common_body = X_train["Body Type"].mode()[0]
median_age = X_train["Car_Age"].median()

for col in ["Brand", "Model"]:
    X_train[col] = X_train[col].fillna("Unknown")
    X_test[col] = X_test[col].fillna("Unknown")

X_train["Kilometers"] = X_train["Kilometers"].fillna(median_km).astype(int)
X_test["Kilometers"] = X_test["Kilometers"].fillna(median_km).astype(int)

X_train["Engine Capacity (CC)"] = X_train["Engine Capacity (CC)"].fillna(median_engine)
X_test["Engine Capacity (CC)"] = X_test["Engine Capacity (CC)"].fillna(median_engine)

X_train["Body Type"] = X_train["Body Type"].fillna(most_common_body)
X_test["Body Type"] = X_test["Body Type"].fillna(most_common_body)

X_train["Car_Age"] = X_train["Car_Age"].fillna(median_age)
X_test["Car_Age"] = X_test["Car_Age"].fillna(median_age)


In [37]:
category_options = {col: sorted(X_train[col].unique().tolist()) for col in text_columns}


In [38]:
price_95th = y_train.quantile(0.95)
y_train = y_train.clip(upper=price_95th)
y_test = y_test.clip(upper=price_95th)


In [39]:
y_train_log = np.log1p(y_train)
y_test_log = np.log1p(y_test)


In [40]:
X_train = pd.get_dummies(X_train, drop_first=True)
X_test = pd.get_dummies(X_test, drop_first=True)
X_test = X_test.reindex(columns=X_train.columns, fill_value=0)

feature_columns = X_train.columns.tolist()


In [41]:
X_train_unscaled = X_train.copy()
X_test_unscaled = X_test.copy()

scaler = StandardScaler()
numeric_columns = ["Kilometers", "Engine Capacity (CC)", "Car_Age"]

X_train[numeric_columns] = scaler.fit_transform(X_train[numeric_columns])
X_test[numeric_columns] = scaler.transform(X_test[numeric_columns])


In [42]:
def calculate_metrics_log(y_true_log, y_pred_log):
    y_true = np.expm1(y_true_log)
    y_pred = np.expm1(y_pred_log)

    r2 = r2_score(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)

    return {"R2": r2, "MAE": mae, "MSE": mse, "RMSE": rmse}

def print_metrics(model_name, metrics):
    print(model_name)
    print(f"R2 Score:  {metrics['R2']:.4f}")
    print(f"MAE:       {metrics['MAE']:,.0f} EGP")
    print(f"MSE:       {metrics['MSE']:,.0f} EGP^2")
    print(f"RMSE:      {metrics['RMSE']:,.0f} EGP")


## 1.(Linear Regression)

In [43]:
lr = LinearRegression()
lr.fit(X_train, y_train_log)
y_pred_lr = lr.predict(X_test)
metrics_lr = calculate_metrics_log(y_test_log, y_pred_lr)
print_metrics("Linear Regression", metrics_lr)


Linear Regression
R2 Score:  0.9096
MAE:       154,904 EGP
MSE:       110,951,236,016 EGP^2
RMSE:      333,093 EGP




```
# This is formatted as code
```

## 2. (Decision Tree)

In [44]:
dt = DecisionTreeRegressor(
    max_depth=10,
    min_samples_split=10,
    min_samples_leaf=5,
    random_state=42
)
dt.fit(X_train_unscaled, y_train_log)
y_pred_dt = dt.predict(X_test_unscaled)
metrics_dt = calculate_metrics_log(y_test_log, y_pred_dt)
print_metrics("Decision Tree", metrics_dt)


Decision Tree
R2 Score:  0.8516
MAE:       213,950 EGP
MSE:       182,124,018,411 EGP^2
RMSE:      426,760 EGP


## 3.
(Random Forest)

In [45]:
rf = RandomForestRegressor(
    n_estimators=200,
    max_depth=12,
    min_samples_split=10,
    min_samples_leaf=5,
    random_state=42
)
rf.fit(X_train_unscaled, y_train_log)
y_pred_rf = rf.predict(X_test_unscaled)
metrics_rf = calculate_metrics_log(y_test_log, y_pred_rf)
print_metrics("Random Forest", metrics_rf)


Random Forest
R2 Score:  0.8807
MAE:       193,258 EGP
MSE:       146,337,220,330 EGP^2
RMSE:      382,540 EGP


## 4. XGBoost

In [46]:
xgb = XGBRegressor(
    n_estimators=500,
    learning_rate=0.03,
    max_depth=8,
    min_child_weight=3,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=1,
    random_state=42
)
xgb.fit(X_train_unscaled, y_train_log)
y_pred_xgb = xgb.predict(X_test_unscaled)
metrics_xgb = calculate_metrics_log(y_test_log, y_pred_xgb)
print_metrics("XGBoost", metrics_xgb)


XGBoost
R2 Score:  0.9198
MAE:       149,452 EGP
MSE:       98,391,294,588 EGP^2
RMSE:      313,674 EGP


In [47]:
xgb.save_model("car_price_model.json")
joblib.dump(scaler, "scaler.pkl")
joblib.dump(feature_columns, "feature_columns.pkl")
joblib.dump(category_options, "category_options.pkl")
preprocessing_stats = {
    "median_km": median_km,
    "median_engine": median_engine,
    "most_common_body": most_common_body,
    "median_age": median_age,
    "price_95th": price_95th,
}
joblib.dump(preprocessing_stats, "preprocessing_stats.pkl")

print("done")


done


In [50]:
from google.colab import drive
drive.mount('/content/drive')

MessageError: Error: credential propagation was unsuccessful